In [1]:
from datetime import datetime
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, coalesce

# Initialize Spark Session
spark = SparkSession.builder.getOrCreate()

today_str = datetime.now().strftime("%Y-%m-%d")
table_price = "int_price_transformed"
table_news = "int_news_transformed"
target_table = "mart_daily_stock_summary"

print(f"Starting Gold Layer Join for {today_str}...")

try:
    # 1. Read the transformed staging tables
    df_price = spark.read.format("delta").table(table_price)
    df_news = spark.read.format("delta").table(table_news)

    # 2. Filter for today's batch to keep the join incredibly fast in memory
    df_price_today = df_price.filter(col("extract_date") == today_str)
    df_news_today = df_news.filter(col("extract_date") == today_str)

    # 3. Perform the Full Outer Join
    df_joined = df_price_today.join(
        df_news_today,
        on=["extract_date", "symbol"],
        how="full"
    )

    # 4. Clean up Nulls caused by the Outer Join
    # If there was no news for a stock today, replace the resulting null counts with 0
    news_count_cols = [
        "total_articles", "bullish_count", "somewhat_bullish_count", 
        "neutral_count", "somewhat_bearish_count", "bearish_count"
    ]
    for c in news_count_cols:
        # coalesce checks if the column is null; if it is, it inserts a literal 0
        df_joined = df_joined.withColumn(c, coalesce(col(c), lit(0)))

    # 5. The Time & Resource Saver Logic (Pre-Filter and Fast Append)
    if spark.catalog.tableExists(target_table):
        
        # A. Query to see which symbols we already joined today
        loaded_df = spark.sql(f"SELECT DISTINCT symbol FROM {target_table} WHERE extract_date = '{today_str}'")
        loaded_symbols = [row['symbol'] for row in loaded_df.collect()]
        
        # B. Filter out already processed symbols
        if loaded_symbols:
            print(f"Symbols already joined for {today_str}: {loaded_symbols}")
            df_joined = df_joined.filter(~col("symbol").isin(loaded_symbols))
            
        # C. Check if we have any data left to write
        if df_joined.isEmpty():
            print(f"⏭️ Joined data already exists for {today_str}. Skipping write!")
        else:
            new_record_count = df_joined.count()
            print(f"Executing fast APPEND for {new_record_count} joined records...")
            
            (df_joined.write
                .format("delta")
                .mode("append")
                .saveAsTable(target_table))
            
            print(f"✅ Successfully appended joined data to {target_table}")

    else:
        # First-time run: Create the table
        print(f"Table does not exist. Creating {target_table}...")
        (df_joined.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(target_table))
        print(f"✅ Successfully created {target_table}")

    # Display the final Gold row(s) for today
    display(spark.sql(f"SELECT * FROM {target_table} WHERE extract_date = '{today_str}'"))

except Exception as e:
    print(f"❌ Join transformation failed. Error: {e}")

StatementMeta(, 28843625-d2de-49ee-99fc-b66f35cc05ce, 3, Finished, Available, Finished, False)

Starting Gold Layer Join for 2026-06-17...
Executing fast APPEND for 3 joined records...
✅ Successfully appended joined data to mart_daily_stock_summary


SynapseWidget(Synapse.DataFrame, abc4c54a-ca02-4a1c-9b6e-cbc1ac048394)